[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [asyncpg and psycopg3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)

# Types and Adaptation &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's Setup, with the `prices` table and the `Money` class. Run it first.
The tasks can be run in any order.


In [1]:
import datetime
import getpass
import json
import os
import subprocess
import sys
import time
from decimal import Decimal
from importlib.metadata import PackageNotFoundError, version

try:
    if version("psycopg") < "3.3" or version("asyncpg") < "0.31":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "psycopg[binary,pool]==3.3.6", "psycopg-pool==3.3.2", "asyncpg==0.31.0"],
                   check=True)

import asyncpg
import psycopg
from psycopg.adapt import Dumper, Loader

def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(database="postgres"):
    """Whether a server is there, asked the only way that needs no client binaries."""
    try:
        with psycopg.connect(f"dbname={database}", connect_timeout=2):
            return True
    except psycopg.OperationalError:
        return False


def start_server(wait=60):
    """Install and start PostgreSQL if nothing is answering. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No PostgreSQL is answering. Start your own server and run this again: "
                           "this cell only installs one on Linux, which is what Colab runs.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"{sudo}apt-get -qq update")
    shell(f"{sudo}apt-get -qq -y install postgresql postgresql-contrib")
    shell(f"{sudo}service postgresql start")                        # Colab has no systemd

    for attempt in range(1, wait + 1):                              # start returns before it listens
        if shell("pg_isready -q")[0] == 0:
            break
        print(f"  waiting for the cluster ({attempt})")              # a silent minute looks hung
        time.sleep(1)
    else:
        raise RuntimeError(f"PostgreSQL did not accept connections within {wait} seconds.")

    me = getpass.getuser()                                          # peer authentication wants a role
    asking = f"""sudo -u postgres psql -tAc "SELECT 1 FROM pg_roles WHERE rolname='{me}'" """
    if shell(asking)[1] != "1":                                     # named for the operating system user
        shell(f"sudo -u postgres createuser -s {me}")
    return "installed and started"

def build(rows=5000):
    """Make the guide database and its events table, and fill it once."""
    with psycopg.connect("dbname=postgres", autocommit=True) as conn:
        if not conn.execute("SELECT 1 FROM pg_database WHERE datname = 'guide'").fetchone():
            conn.execute("CREATE DATABASE guide")                   # cannot run in a transaction

    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        for (leftover,) in conn.execute(                            # whatever an earlier run made
                "SELECT tablename FROM pg_tables "
                "WHERE schemaname = 'public' AND tablename <> 'events'").fetchall():
            conn.execute(f'DROP TABLE IF EXISTS "{leftover}" CASCADE')

        conn.execute("""CREATE TABLE IF NOT EXISTS events (
                            id bigserial PRIMARY KEY,
                            ts timestamptz NOT NULL DEFAULT now(),
                            kind text NOT NULL,
                            payload jsonb NOT NULL)""")
        if conn.execute("SELECT count(*) FROM events").fetchone()[0] == 0:
            conn.execute("""INSERT INTO events (kind, payload)
                            SELECT (ARRAY['click', 'view', 'purchase'])[1 + n %% 3],
                                   jsonb_build_object('n', n, 'size', 1 + n %% 7)
                            FROM generate_series(1, %s) AS n""", (rows,))
        return conn.execute("SELECT count(*) FROM events").fetchone()[0]

def report():
    """One line naming what this notebook is running against."""
    rows = build()                                                  # makes the database if it is new
    with psycopg.connect("dbname=guide") as conn:
        major = int(conn.execute("SHOW server_version_num").fetchone()[0]) // 10000
    return (f"PostgreSQL {major} | psycopg {version('psycopg')} | asyncpg {version('asyncpg')} "
            f"| events: {rows} rows")

class Money:
    """A class of your own, which the drivers have never heard of."""

    def __init__(self, amount):
        self.amount = Decimal(amount)

    def __repr__(self):
        return f"Money({self.amount})"


def build_prices():
    """One row with a column of each type this notebook is about."""
    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        conn.execute("DROP TABLE IF EXISTS prices")
        conn.execute("CREATE TABLE prices (id int, amount numeric(10,2), "
                     "at timestamptz, tags text[])")
        conn.execute("INSERT INTO prices VALUES "
                     "(1, 19.99, '2026-01-15 12:00:00+00', ARRAY['sea', 'stone'])")


def utc(**kwargs):
    """A connection whose session time zone is UTC, so a printed timestamp is the same everywhere."""
    return psycopg.connect("dbname=guide", options="-c TimeZone=UTC", **kwargs)


print("server:", start_server())
print(report())
build_prices()
print("prices is ready")


server: already running
PostgreSQL 16 | psycopg 3.3.6 | asyncpg 0.31.0 | events: 5000 rows
prices is ready


**1.** What the server makes of five values.


In [2]:
with psycopg.connect("dbname=guide") as conn:
    for value in (7, 7.5, True, Decimal("2.50"), [1, 2]):
        named = conn.execute("SELECT pg_typeof(%s)::text", (value,)).fetchone()[0]
        print(f"  {type(value).__name__:<8} {str(value):<10} -> {named}")


  int      7          -> smallint
  float    7.5        -> double precision
  bool     True       -> boolean
  Decimal  2.50       -> numeric
  list     [1, 2]     -> smallint[]


`pg_typeof` works here because each of these five carries enough information for the server to name
a type. A bare string or `None` does not, and asking the same question about one of those gives
`could not determine data type of parameter $1`, which is the server saying it has nothing to go on.


**2.** An exact number meeting an inexact one.


In [3]:
with psycopg.connect("dbname=guide") as conn:
    amount = conn.execute("SELECT amount FROM prices WHERE id = 1").fetchone()[0]

print("type: ", type(amount).__name__, "| value:", amount)

try:
    amount * 1.2
except TypeError as error:
    print("times a float:  ", error)

print("times a Decimal:", amount * Decimal("1.2"))


type:  Decimal | value: 19.99
times a float:   unsupported operand type(s) for *: 'decimal.Decimal' and 'float'
times a Decimal: 23.988


Python refuses to mix them rather than quietly making the result approximate. For money that refusal
is the feature: converting to `float` to get past it reintroduces exactly the rounding the `numeric`
column was chosen to avoid.


**3.** One instant, two zones.


In [4]:
readings = {}
for zone in ("UTC", "Europe/Dublin"):
    with psycopg.connect("dbname=guide", options=f"-c TimeZone={zone}") as conn:
        readings[zone] = conn.execute("SELECT at FROM prices WHERE id = 1").fetchone()[0]

for zone, moment in readings.items():
    print(f"  {zone:<14} {moment.isoformat()}")
print("equal:", readings["UTC"] == readings["Europe/Dublin"])


  UTC            2026-01-15T12:00:00+00:00
  Europe/Dublin  2026-01-15T12:00:00+00:00
equal: True


Two strings, one moment. `timestamptz` stores the instant and renders it in whatever zone the session
is using, so the difference is in the printing rather than in the data.


**4.** A rule for a class of your own.


In [5]:
class Weight:
    def __init__(self, grams):
        self.grams = int(grams)

    def __repr__(self):
        return f"Weight({self.grams}g)"


class WeightDumper(Dumper):
    oid = psycopg.adapters.types["int4"].oid

    def dump(self, obj):
        return str(obj.grams).encode()


with psycopg.connect("dbname=guide") as conn:
    try:
        conn.execute("SELECT %s", (Weight(250),))
    except psycopg.ProgrammingError as error:
        print("before:", error)
    conn.rollback()

    conn.adapters.register_dumper(Weight, WeightDumper)
    print("after: ", conn.execute("SELECT %s", (Weight(250),)).fetchone())


before: cannot adapt type 'Weight' using placeholder '%s' (format: AUTO)
after:  (250,)


The `oid` is the whole of what the rule tells the server: these bytes are an `int4`. `dump` returns
bytes because bytes are the only thing the socket carries.


**5.** A list as an array.


In [6]:
with psycopg.connect("dbname=guide") as conn:
    for value in ([1, 2, 3], ["a", "b"], [[1, 2], [3, 4]]):
        back = conn.execute("SELECT %s", (value,)).fetchone()[0]
        print(f"  {str(value):<16} -> {type(back).__name__:<6} {back}")

    print("  and a list of numbers is an array:",
          conn.execute("SELECT pg_typeof(%s)::text", ([1, 2, 3],)).fetchone()[0])


  [1, 2, 3]        -> list   [1, 2, 3]
  ['a', 'b']       -> str    {a,b}
  [[1, 2], [3, 4]] -> list   [[1, 2], [3, 4]]
  and a list of numbers is an array: smallint[]


A list is an array, and a list of lists is still one array, with two dimensions. That rule is what
lets `= ANY(%s)` take a Python list without anything being formatted into the statement.

`pg_typeof` is asked only about the numbers, because a list of strings has the same problem a bare
string has: nothing in it tells the server which type to name, and it answers
`could not determine data type of parameter $1`. The round trip has no such problem, since by then
the value has been through a column or a cast.


**6.** JSONB through asyncpg, before and after.


In [7]:
conn = await asyncpg.connect(database="guide")

before = await conn.fetchval("SELECT payload FROM events ORDER BY id LIMIT 1")
print("before:", type(before).__name__, repr(before))

await conn.set_type_codec("jsonb", encoder=json.dumps, decoder=json.loads, schema="pg_catalog")

after = await conn.fetchval("SELECT payload FROM events ORDER BY id LIMIT 1")
print("after: ", type(after).__name__, after)
print("and reading a key works:", after["n"])
await conn.close()


before: str '{"n": 1, "size": 2}'
after:  dict {'n': 1, 'size': 2}
and reading a key works: 1


The column did not change and neither did the row. What changed is that the connection now has a
decoder for that type, so the bytes are turned into a dictionary instead of being handed over as the
text they are stored as.


---

&#8592; **Back to:** [Types and Adaptation](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/05-types-and-adaptation.ipynb)  &nbsp;&middot;&nbsp;  [asyncpg and psycopg3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)
